# LeetCode 2586 — Count the Number of Vowel Strings in Range bằng Python

Link đề: https://leetcode.com/problems/count-the-number-of-vowel-strings-in-range/ · Độ khó: **Easy** · Chủ đề: Array, String, Counting

Skeleton LeetCode:

```python
class Solution:
    def vowelStrings(self, words: List[str], left: int, right: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu "vowel string" (xâu nguyên âm): xâu **bắt đầu và kết thúc** bằng một nguyên âm `a, e, i, o, u`; và chỉ đếm các xâu có chỉ số trong đoạn **đóng** `[left, right]`.
2. Dùng chỉ số `w[0]` (ký tự đầu), `w[-1]` (ký tự cuối) và toán tử `in` để kiểm tra một ký tự có thuộc tập nguyên âm.
3. Cắt đoạn `words[left:right + 1]` (nhớ `+ 1` vì chặn phải của slicing là **mở**) và đếm bằng `sum` trên giá trị `True/False`.
4. Dùng `str.startswith` / `str.endswith` với **tuple** nhiều tiền tố, và kỹ thuật **prefix count** (đếm tiền tố) để đếm đoạn trong `O(1)` cho nhiều truy vấn.
5. Viết 4 cách và đối chiếu bằng test ngẫu nhiên có seed.
6. Tránh các lỗi quen thuộc: `words[left:right]` thiếu phần tử cuối, chỉ kiểm tra một đầu, dùng `or` thay `and`, lỗi độ ưu tiên `w[0] and w[-1] in V`, bỏ qua khoảng `[left, right]`.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words` (chỉ số bắt đầu từ 0) và hai số nguyên `left`, `right`.

Một chuỗi gọi là **vowel string** nếu nó **bắt đầu bằng** một nguyên âm **và kết thúc bằng** một nguyên âm, với nguyên âm là một trong `'a'`, `'e'`, `'i'`, `'o'`, `'u'`.

Trả về **số lượng vowel string** `words[i]` với `i` thuộc đoạn **đóng** `[left, right]` (gồm cả `left` và `right`).

| Ví dụ | words | left | right | Output | Giải thích |
|---|---|---|---|---|---|
| 1 | `["are","amy","u"]` | 0 | 2 | 2 | `"are"` (a…e) đúng; `"amy"` kết thúc bằng `y` nên sai; `"u"` (u…u) đúng |
| 2 | `["hey","aeo","mu","ooo","artro"]` | 1 | 4 | 3 | chỉ xét chỉ số 1..4: `"aeo"`, `"ooo"`, `"artro"` đúng; `"mu"` bắt đầu bằng `m` nên sai |

Constraints: `1 <= words.length <= 1000`; `1 <= words[i].length <= 10`; `words[i]` chỉ gồm chữ cái thường tiếng Anh; `0 <= left <= right < words.length`.

In [1]:
words = ["hey", "aeo", "mu", "ooo", "artro"]
left, right = 1, 4
for i in range(left, right + 1):
    w = words[i]
    print(f"i={i}: {w!r} bắt đầu {w[0]!r}, kết thúc {w[-1]!r}")
print("theo đề: aeo, ooo, artro là vowel string -> 3")

i=1: 'aeo' bắt đầu 'a', kết thúc 'o'
i=2: 'mu' bắt đầu 'm', kết thúc 'u'
i=3: 'ooo' bắt đầu 'o', kết thúc 'o'
i=4: 'artro' bắt đầu 'a', kết thúc 'o'
theo đề: aeo, ooo, artro là vowel string -> 3


## 3. Kiến thức nền

### 3.1. Chỉ số ký tự: `w[0]` và `w[-1]`

Chuỗi cũng đánh chỉ số như list: `w[0]` là ký tự **đầu**, `w[len(w) - 1]` là ký tự **cuối**. Python cho viết tắt `w[-1]` = ký tự cuối, `w[-2]` = ký tự kế cuối. Với chuỗi **một** ký tự thì `w[0]` và `w[-1]` là **cùng một ký tự**.

In [2]:
w = "artro"
print(w[0], w[-1], w[len(w) - 1])     # a o o
u = "u"
print(u[0], u[-1], u[0] == u[-1])     # u u True

a o o
u u True


### 3.2. Toán tử `in` với chuỗi và với `set`

`x in "aeiou"` kiểm tra `x` có là **chuỗi con** của `"aeiou"` hay không. Với ký tự đơn thì tương đương "x là một trong 5 nguyên âm". Dùng `set` (`{'a','e','i','o','u'}`) tra cứu nhanh trung bình `O(1)` và an toàn hơn vì không bị nhầm với chuỗi con nhiều ký tự (ví dụ `"ae" in "aeiou"` là `True`).

In [3]:
vowels = "aeiou"
vowel_set = {'a', 'e', 'i', 'o', 'u'}
print('a' in vowels, 'y' in vowels)
print("ae" in vowels, "ae" in vowel_set)     # True False  -> chuỗi con vs phần tử

True False
True False


### 3.3. Slicing đoạn `[left, right]` đóng

`words[a:b]` lấy chỉ số `a` tới `b - 1` — chặn phải **mở**. Để lấy cả `right` phải viết `words[left:right + 1]`.

In [4]:
words = ["w0", "w1", "w2", "w3", "w4"]
left, right = 1, 3
print(words[left:right])          # ['w1', 'w2']        THIẾU w3
print(words[left:right + 1])      # ['w1', 'w2', 'w3']  đúng đoạn đóng

['w1', 'w2']
['w1', 'w2', 'w3']


### 3.4. `True`/`False` là số 1/0 — dùng `sum` để đếm

Trong Python `True == 1` và `False == 0`. Vì vậy `sum(điều_kiện for x in ...)` đếm số phần tử thoả điều kiện.

In [5]:
nums = [3, 8, 5, 10, 7]
print(sum(x > 4 for x in nums))       # 4
print(sum([True, False, True]))       # 2

4
2


### 3.5. `startswith` / `endswith` với tuple

`s.startswith(("a", "e"))` đúng nếu `s` bắt đầu bằng **một trong** các tiền tố trong tuple. Tương tự `endswith`. Cách này gọn hơn nhiều so với chuỗi `or` dài.

In [6]:
V = ("a", "e", "i", "o", "u")
for w in ["are", "amy", "u", "hey"]:
    print(w, w.startswith(V), w.endswith(V), w.startswith(V) and w.endswith(V))

are True True True
amy True False False
u True True True
hey False False False


### 3.6. Prefix count — đếm trong đoạn bằng hiệu hai tiền tố

Nếu `pref[i]` là **số** phần tử thoả điều kiện trong `words[0..i-1]` (với `pref[0] = 0`), thì số phần tử thoả trong đoạn `[left, right]` là `pref[right + 1] - pref[left]`. Dựng `pref` một lần, rồi mỗi truy vấn đoạn tốn `O(1)`.

In [7]:
flags = [1, 0, 1, 1, 0]               # 1 = thoả, 0 = không
pref = [0]
for f in flags:
    pref.append(pref[-1] + f)
print(pref)                            # [0, 1, 1, 2, 3, 3]
left, right = 1, 3
print("trong [1, 3]:", pref[right + 1] - pref[left], "(flags[1..3] =", flags[left:right + 1], ")")

[0, 1, 1, 2, 3, 3]
trong [1, 3]: 2 (flags[1..3] = [0, 1, 1] )


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm đúng như đề: đi qua từng chuỗi **trong đoạn** `[left, right]`, nhìn chữ đầu và chữ cuối; nếu cả hai là nguyên âm thì đếm.

Ví dụ 2: `words = ["hey","aeo","mu","ooo","artro"]`, đoạn `[1, 4]`:

```
i=1 "aeo"   : a ... o  -> cả hai là nguyên âm  -> +1
i=2 "mu"    : m ... u  -> m không phải nguyên âm -> không đếm
i=3 "ooo"   : o ... o  -> +1
i=4 "artro" : a ... o  -> +1
tổng = 3     (i = 0 là "hey" nằm ngoài đoạn nên không xét)
```

In [8]:
words = ["hey", "aeo", "mu", "ooo", "artro"]
left, right = 1, 4
count = 0
for i in range(left, right + 1):
    w = words[i]
    ok = w[0] in "aeiou" and w[-1] in "aeiou"
    print(f"i={i} {w!r}: đầu {w[0]!r} cuối {w[-1]!r} -> {'đếm' if ok else 'bỏ'}")
    if ok:
        count += 1
print("đáp án:", count)

i=1 'aeo': đầu 'a' cuối 'o' -> đếm
i=2 'mu': đầu 'm' cuối 'u' -> bỏ
i=3 'ooo': đầu 'o' cuối 'o' -> đếm
i=4 'artro': đầu 'a' cuối 'o' -> đếm
đáp án: 3


## 5. Cách 1 — Brute Force / baseline

Duyệt **toàn bộ** mảng, mỗi phần tử kiểm tra "có nằm trong đoạn không" và "có phải vowel string không" bằng các phép so sánh trực tiếp (liệt kê từng nguyên âm).

In [9]:
from typing import List

def vowel_strings_bruteforce(words: List[str], left: int, right: int) -> int:
    count = 0
    for i in range(len(words)):
        if i < left or i > right:
            continue                          # ngoài đoạn [left, right]
        w = words[i]
        first, last = w[0], w[len(w) - 1]
        first_ok = first == 'a' or first == 'e' or first == 'i' or first == 'o' or first == 'u'
        last_ok = last == 'a' or last == 'e' or last == 'i' or last == 'o' or last == 'u'
        if first_ok and last_ok:
            count += 1
    return count

In [10]:
print(vowel_strings_bruteforce(["are", "amy", "u"], 0, 2))                      # 2
print(vowel_strings_bruteforce(["hey", "aeo", "mu", "ooo", "artro"], 1, 4))     # 3

2
3


**Độ phức tạp:** gọi `n = len(words)`, `L` là độ dài chuỗi tối đa (≤ 10; ta chỉ nhìn 2 ký tự nên mỗi chuỗi tốn `O(1)`). Time `O(n)`; Space `O(1)`.

**Nút thắt:** về mặt thời gian đây gần như đã tối ưu cho **một** truy vấn (phải nhìn mỗi chuỗi trong đoạn ít nhất một lần). Chỗ chưa hay: (a) duyệt cả phần **ngoài đoạn** rồi mới `continue`; (b) chuỗi `or` lặp lại 5 lần x 2; (c) nếu có **nhiều** truy vấn `(left, right)` thì mỗi lần lại duyệt lại.

## 6. Ý tưởng thuật toán / bước đột phá

Ba cải tiến nhỏ nhưng đáng học:

1. **Chỉ cắt đúng đoạn cần xét:** `words[left:right + 1]` thay vì duyệt cả mảng rồi `continue`.
2. **Gom "nguyên âm" vào một tập** (`"aeiou"` hoặc `set`) hoặc dùng `startswith/endswith` với tuple, để điều kiện là một dòng.
3. **Prefix count:** tính trước `pref[i]`; với nhiều truy vấn, mỗi đoạn trả lời `pref[right + 1] - pref[left]` trong `O(1)`.

Quy tắc "nếu… thì…": `w[0] in V` **và** `w[-1] in V` → tính 1; ngược lại 0. Tổng các 0/1 trong đoạn `[left, right]` là đáp án.

In [11]:
words = ["hey", "aeo", "mu", "ooo", "artro"]
V = "aeiou"
flags = [int(w[0] in V and w[-1] in V) for w in words]
print("flags toàn mảng:", flags)
print("trong [1, 4]:", sum(flags[1:4 + 1]))

flags toàn mảng: [0, 1, 0, 1, 1]
trong [1, 4]: 3


## 7. Các cách tiếp theo

Ta có thêm 3 cách: **(2)** slicing + `for` với tập nguyên âm, **(3)** `startswith`/`endswith` + `sum` một dòng, **(4)** prefix count (tối ưu cho nhiều truy vấn).

### Cách 2 — Cắt đoạn + `for` với `set`

Giải thích từng đoạn:

1. `vowels = set("aeiou")` — tập 5 nguyên âm; `set("aeiou")` tự tách chuỗi thành 5 ký tự.
2. `for w in words[left:right + 1]` — chỉ duyệt đúng đoạn đóng `[left, right]`.
3. Nếu `w[0] in vowels and w[-1] in vowels` thì `count += 1`.

In [12]:
def vowel_strings_slice(words, left, right):
    vowels = set("aeiou")
    count = 0
    for w in words[left:right + 1]:
        if w[0] in vowels and w[-1] in vowels:
            count += 1
    return count

print(vowel_strings_slice(["are", "amy", "u"], 0, 2))
print(vowel_strings_slice(["hey", "aeo", "mu", "ooo", "artro"], 1, 4))

2
3


In [13]:
from typing import List

class SolutionSlice:
    def vowelStrings(self, words: List[str], left: int, right: int) -> int:
        vowels = set("aeiou")
        count = 0
        for w in words[left:right + 1]:
            if w[0] in vowels and w[-1] in vowels:
                count += 1
        return count

Độ phức tạp Cách 2: `words[left:right + 1]` tạo bản sao `O(m)` với `m = right - left + 1`; duyệt `O(m)` → Time `O(m)`; Space `O(m)` cho lát cắt (có thể tránh bằng `range(left, right + 1)`).

### Cách 3 — `startswith` / `endswith` + `sum` (một dòng)

`V = ("a","e","i","o","u")` là tuple. Biểu thức `w.startswith(V) and w.endswith(V)` là `True`/`False`; `sum(...)` đếm số `True`.

In [14]:
def vowel_strings_oneliner(words, left, right):
    V = ("a", "e", "i", "o", "u")
    return sum(w.startswith(V) and w.endswith(V) for w in words[left:right + 1])

print(vowel_strings_oneliner(["are", "amy", "u"], 0, 2))
print(vowel_strings_oneliner(["hey", "aeo", "mu", "ooo", "artro"], 1, 4))

2
3


In [15]:
from typing import List

class SolutionOneLiner:
    def vowelStrings(self, words: List[str], left: int, right: int) -> int:
        V = ("a", "e", "i", "o", "u")
        return sum(w.startswith(V) and w.endswith(V) for w in words[left:right + 1])

Độ phức tạp Cách 3: Time `O(m)`; Space `O(m)` (lát cắt); generator trong `sum` không tạo thêm list.

### Cách 4 — Prefix count (đếm tiền tố)

Dựng `pref` với `pref[i + 1] = pref[i] + (words[i] là vowel string)`. Đáp án cho `[left, right]` là `pref[right + 1] - pref[left]`. Phần dựng tốn `O(n)` **một lần**; mỗi truy vấn sau đó `O(1)` — đó chính là bài "Count Vowel Strings in Ranges" (2559) nâng cấp.

In [16]:
def build_prefix(words):
    V = set("aeiou")
    pref = [0]
    for w in words:
        pref.append(pref[-1] + (1 if w[0] in V and w[-1] in V else 0))
    return pref

def vowel_strings_prefix(words, left, right):
    pref = build_prefix(words)
    return pref[right + 1] - pref[left]

ws = ["hey", "aeo", "mu", "ooo", "artro"]
print(build_prefix(ws))
print(vowel_strings_prefix(ws, 1, 4), vowel_strings_prefix(["are", "amy", "u"], 0, 2))

[0, 0, 1, 1, 2, 3]
3 2


In [17]:
from typing import List

class SolutionPrefix:
    def vowelStrings(self, words: List[str], left: int, right: int) -> int:
        V = set("aeiou")
        pref = [0]
        for w in words:
            pref.append(pref[-1] + (1 if w[0] in V and w[-1] in V else 0))
        return pref[right + 1] - pref[left]

Độ phức tạp Cách 4: Time `O(n)` dựng + `O(1)` mỗi truy vấn; Space `O(n)`. Với **một** truy vấn như bài này, nó không nhanh hơn Cách 2 (thậm chí duyệt cả mảng), nhưng nó là công cụ chuẩn khi có **nhiều** truy vấn.

## 8. TRACE từng bước

Trace Cách 2 với `words = ["hey","aeo","mu","ooo","artro"]`, `left = 1`, `right = 4`.

| `i` | `w` | `w[0]` | `w[-1]` | cả hai là nguyên âm? | `count` sau |
|---|---|---|---|---|---|
| 1 | `aeo` | `a` | `o` | có | 1 |
| 2 | `mu` | `m` | `u` | không (`m`) | 1 |
| 3 | `ooo` | `o` | `o` | có | 2 |
| 4 | `artro` | `a` | `o` | có | 3 |

Chuỗi ở chỉ số 0 (`"hey"`) nằm ngoài đoạn nên không xuất hiện trong bảng.

In [18]:
def trace_vowel_strings(words, left, right):
    vowels = set("aeiou")
    count = 0
    for i in range(left, right + 1):
        w = words[i]
        ok = w[0] in vowels and w[-1] in vowels
        if ok:
            count += 1
        print(f"i={i} w={w!r:8} đầu={w[0]!r} cuối={w[-1]!r} -> {'có ' if ok else 'không'} | count={count}")
    return count

trace_vowel_strings(["hey", "aeo", "mu", "ooo", "artro"], 1, 4)
trace_vowel_strings(["are", "amy", "u"], 0, 2)

i=1 w='aeo'    đầu='a' cuối='o' -> có  | count=1
i=2 w='mu'     đầu='m' cuối='u' -> không | count=1
i=3 w='ooo'    đầu='o' cuối='o' -> có  | count=2
i=4 w='artro'  đầu='a' cuối='o' -> có  | count=3
i=0 w='are'    đầu='a' cuối='e' -> có  | count=1
i=1 w='amy'    đầu='a' cuối='y' -> không | count=1
i=2 w='u'      đầu='u' cuối='u' -> có  | count=2


2

In [19]:
# Trace prefix count: dựng pref rồi trả lời hai đoạn khác nhau
ws = ["hey", "aeo", "mu", "ooo", "artro"]
pref = [0]
for i, w in enumerate(ws):
    flag = 1 if w[0] in "aeiou" and w[-1] in "aeiou" else 0
    pref.append(pref[-1] + flag)
    print(f"i={i} {w!r}: flag={flag} -> pref={pref}")
for l, r in [(1, 4), (0, 0), (2, 3)]:
    print(f"đoạn [{l}, {r}] = pref[{r + 1}] - pref[{l}] = {pref[r + 1]} - {pref[l]} = {pref[r + 1] - pref[l]}")

i=0 'hey': flag=0 -> pref=[0, 0]
i=1 'aeo': flag=1 -> pref=[0, 0, 1]
i=2 'mu': flag=0 -> pref=[0, 0, 1, 1]
i=3 'ooo': flag=1 -> pref=[0, 0, 1, 1, 2]
i=4 'artro': flag=1 -> pref=[0, 0, 1, 1, 2, 3]
đoạn [1, 4] = pref[5] - pref[1] = 3 - 0 = 3
đoạn [0, 0] = pref[1] - pref[0] = 0 - 0 = 0
đoạn [2, 3] = pref[4] - pref[2] = 2 - 1 = 1


## 9. Code nộp LeetCode

Nộp Cách 2 (cắt đoạn + `for` + `set`): rõ ràng, dễ giải thích, `O(m)` thời gian.

In [20]:
from typing import List

class Solution:
    def vowelStrings(self, words: List[str], left: int, right: int) -> int:
        vowels = set("aeiou")
        count = 0
        for i in range(left, right + 1):
            w = words[i]
            if w[0] in vowels and w[-1] in vowels:
                count += 1
        return count

print(Solution().vowelStrings(["are", "amy", "u"], 0, 2))                      # 2
print(Solution().vowelStrings(["hey", "aeo", "mu", "ooo", "artro"], 1, 4))     # 3

2
3


## 10. Vì sao thuật toán đúng

- **Đúng đoạn:** `range(left, right + 1)` sinh đúng các chỉ số `left, left+1, ..., right` (đoạn đóng), mỗi chỉ số một lần. Vì `0 <= left <= right < len(words)` nên mọi `words[i]` đều tồn tại.
- **Đúng điều kiện:** theo đề, `words[i]` là vowel string ⇔ `words[i][0]` là nguyên âm **và** `words[i][-1]` là nguyên âm. Ký tự đầu/cuối luôn tồn tại vì mỗi chuỗi có ít nhất 1 ký tự. Với chuỗi 1 ký tự thì đầu = cuối, nên nó là vowel string ⇔ chính ký tự đó là nguyên âm.
- **Đếm đủ, không trùng:** mỗi chỉ số trong đoạn được xét đúng một lần và cộng 1 đúng khi thoả điều kiện ⇒ `count` là số vowel string trong đoạn.
- **Prefix count (Cách 4):** `pref[k]` = số vowel string trong `words[0..k-1]` (quy nạp: `pref[0] = 0`; thêm `words[k]` làm tăng 1 nếu thoả). Số trong `[left, right]` = số trong `[0, right]` trừ số trong `[0, left - 1]` = `pref[right + 1] - pref[left]`.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(words)`, `m = right - left + 1` (số chuỗi trong đoạn). Vì độ dài mỗi chuỗi `<= 10` và ta chỉ nhìn 2 ký tự, kiểm tra một chuỗi là `O(1)`.

| Cách | Time | Space (phụ) |
|---|---|---|
| 1. Duyệt cả mảng, `continue` ngoài đoạn | `O(n)` | `O(1)` |
| 2. Cắt đoạn + `for` + `set` (hoặc `range`) | `O(m)` | `O(m)` nếu cắt lát; `O(1)` nếu dùng `range` |
| 3. `startswith/endswith` + `sum` | `O(m)` | `O(m)` (lát cắt) |
| 4. Prefix count | `O(n)` dựng, `O(1)` / truy vấn | `O(n)` |

Với một truy vấn và `n <= 1000`, mọi cách tức thì.

## 12. Bộ test

Test 1: ví dụ đề và các edge case: chuỗi một ký tự (nguyên âm / phụ âm), đoạn chỉ một phần tử, đoạn toàn mảng, không có vowel string nào, tất cả đều là vowel string, chuỗi dài 10.

In [21]:
impls = [
    ("brute", vowel_strings_bruteforce),
    ("slice", vowel_strings_slice),
    ("oneliner", vowel_strings_oneliner),
    ("prefix", vowel_strings_prefix),
    ("class_slice", SolutionSlice().vowelStrings),
    ("class_oneliner", SolutionOneLiner().vowelStrings),
    ("class_prefix", SolutionPrefix().vowelStrings),
    ("submit", Solution().vowelStrings),
]
tests = [
    (["are", "amy", "u"], 0, 2, 2),
    (["hey", "aeo", "mu", "ooo", "artro"], 1, 4, 3),
    (["a"], 0, 0, 1),                              # một chuỗi một nguyên âm
    (["b"], 0, 0, 0),                              # một chuỗi một phụ âm
    (["a", "b", "e", "z", "u"], 2, 2, 1),          # đoạn chỉ 1 phần tử
    (["a", "b", "e", "z", "u"], 0, 4, 3),          # cả mảng
    (["bcd", "xyz", "hello"], 0, 2, 0),            # không có vowel string
    (["aa", "ee", "ii", "oo", "uu"], 0, 4, 5),     # tất cả đều đúng
    (["abcdefghio", "ebcdefghia", "bbbbbbbbbb"], 0, 2, 2),   # chuỗi dài 10
    (["ab", "ba", "aa", "bb"], 1, 2, 1),           # đầu/cuối chỉ một đúng không tính
    (["ua", "ay", "ya"], 0, 2, 1),                 # 'y' không phải nguyên âm
]
for name, f in impls:
    for words, left, right, expected in tests:
        got = f(list(words), left, right)
        assert got == expected, (name, words, left, right, got, expected)
    print(name, "PASS", len(tests), "test")

brute PASS 11 test
slice PASS 11 test
oneliner PASS 11 test
prefix PASS 11 test
class_slice PASS 11 test
class_oneliner PASS 11 test
class_prefix PASS 11 test
submit PASS 11 test


Test 2: ngẫu nhiên có seed, **đúng constraints**: `1..1000` chuỗi, mỗi chuỗi dài `1..10` chữ cái thường, `0 <= left <= right < n`. Bảng chữ cái được thiên lệch về nguyên âm để có nhiều vowel string. Oracle độc lập: dùng biểu thức chính quy `re`.

In [22]:
import random, re
random.seed(2586)

ALPHABET = "aeiouaeioubcdxyzmnp"           # thiên lệch nguyên âm nhưng vẫn có phụ âm
PATTERN = re.compile(r"^[aeiou](.*[aeiou])?$")   # bắt đầu và kết thúc bằng nguyên âm (kể cả 1 ký tự)

def oracle(words, left, right):
    total = 0
    for i, w in enumerate(words):
        if left <= i <= right and PATTERN.match(w):
            total += 1
    return total

for _ in range(500):
    n = random.choice([1, 2, 5, 30, 200, 1000])
    words = ["".join(random.choice(ALPHABET) for _ in range(random.randint(1, 10))) for _ in range(n)]
    a = random.randint(0, n - 1)
    b = random.randint(0, n - 1)
    left, right = min(a, b), max(a, b)
    expected = oracle(words, left, right)
    for name, f in impls:
        assert f(list(words), left, right) == expected, (name, left, right)
print("500 test ngẫu nhiên: tất cả cách đều khớp oracle (regex)")

500 test ngẫu nhiên: tất cả cách đều khớp oracle (regex)


Test 3: tính chất — đáp án nằm trong `[0, right - left + 1]`; chia đoạn `[left, right]` thành hai đoạn liền kề thì **tổng hai đáp án bằng đáp án cả đoạn**; hàm **không sửa** `words`.

In [23]:
random.seed(3)
for _ in range(300):
    n = random.randint(2, 60)
    words = ["".join(random.choice(ALPHABET) for _ in range(random.randint(1, 10))) for _ in range(n)]
    left = random.randint(0, n - 2)
    right = random.randint(left + 1, n - 1)
    mid = random.randint(left, right - 1)
    keep = list(words)
    whole = Solution().vowelStrings(words, left, right)
    parts = Solution().vowelStrings(words, left, mid) + Solution().vowelStrings(words, mid + 1, right)
    assert 0 <= whole <= right - left + 1
    assert whole == parts
    assert words == keep
print("tính chất cộng đoạn + không sửa input: PASS")

tính chất cộng đoạn + không sửa input: PASS


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `words[left:right]` — thiếu phần tử ở `right`

In [24]:
def wrong_slice(words, left, right):
    V = set("aeiou")
    return sum(w[0] in V and w[-1] in V for w in words[left:right])     # thiếu words[right]

print(wrong_slice(["are", "amy", "u"], 0, 2), "(đáp án đúng là 2)")

1 (đáp án đúng là 2)


Vì sao sai: chặn phải của slicing là **mở**, nên `words[0:2]` chỉ gồm chỉ số `0, 1` và bỏ mất `"u"`. Đề lấy đoạn **đóng** → `words[left:right + 1]`.

### Lỗi 2: chỉ kiểm tra một đầu (hoặc dùng `or` thay `and`)

In [25]:
def wrong_or(words, left, right):
    V = set("aeiou")
    return sum(w[0] in V or w[-1] in V for w in words[left:right + 1])   # SAI: cần "và"

print(wrong_or(["are", "amy", "u"], 0, 2), "(đáp án đúng là 2)")
print(wrong_or(["ab", "ba"], 0, 1), "(đáp án đúng là 0)")

3 (đáp án đúng là 2)
2 (đáp án đúng là 0)


Vì sao sai: `"amy"` bắt đầu bằng nguyên âm `a` nhưng kết thúc bằng `y`; với `or` nó vẫn bị đếm (ra `3` cho ví dụ 1). Vowel string cần **cả hai** đầu là nguyên âm nên dùng `and`.

### Lỗi 3: lỗi độ ưu tiên `w[0] and w[-1] in V`

In [26]:
def wrong_precedence(words, left, right):
    V = set("aeiou")
    return sum(bool(w[0] and w[-1] in V) for w in words[left:right + 1])   # w[0] luôn "truthy"

print(wrong_precedence(["ba"], 0, 0), "(đáp án đúng là 0: 'ba' bắt đầu bằng phụ âm b)")

1 (đáp án đúng là 0: 'ba' bắt đầu bằng phụ âm b)


Vì sao sai: `w[0] and w[-1] in V` được hiểu là `w[0] and (w[-1] in V)`. Ký tự `w[0]` là chuỗi không rỗng nên luôn "đúng" (truthy) — biểu thức chỉ còn kiểm tra ký tự **cuối**. Phải viết đủ hai vế: `w[0] in V and w[-1] in V`.

### Lỗi 4: đếm cả mảng, quên giới hạn `[left, right]`

In [27]:
def wrong_all(words, left, right):
    V = set("aeiou")
    return sum(w[0] in V and w[-1] in V for w in words)       # bỏ qua left, right

print(wrong_all(["hey", "aeo", "mu", "ooo", "artro"], 1, 3), "vs đúng", Solution().vowelStrings(["hey", "aeo", "mu", "ooo", "artro"], 1, 3))

3 vs đúng 2


Vì sao sai: đề chỉ đếm các chuỗi có chỉ số trong `[left, right]`. Ở ví dụ trên, `"artro"` ở chỉ số 4 nằm ngoài đoạn `[1, 3]` nhưng vẫn bị đếm. Luôn nhớ hai tham số `left`, `right` là một phần của đề.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm số phần tử thoả điều kiện trong **đoạn** `[l, r]`" → cắt lát `a[l:r + 1]` rồi `sum`, hoặc **prefix count** nếu có nhiều truy vấn.
- "Bắt đầu và kết thúc bằng…" → `s[0]` và `s[-1]`; luôn nhớ chuỗi 1 ký tự có đầu = cuối.
- Kiểm tra thuộc tập cố định → `set` (hoặc tuple cho `startswith`/`endswith`).
- Nhớ bẫy: `w[0] and w[-1] in V` sai ưu tiên; `"ae" in "aeiou"` là `True`.
- Đoạn đóng `[left, right]` ⇔ `range(left, right + 1)` ⇔ `a[left:right + 1]`: **luôn `+ 1`**.
- Bài mở rộng trực tiếp: "Count Vowel Strings in Ranges" (2559) — nhiều truy vấn, dùng prefix count đúng như Cách 4.

## 15. Biến thể của bài

1. **Nhiều truy vấn** `(left, right)`: dựng prefix count một lần, mỗi truy vấn `O(1)` (bài 2559).
2. Thay điều kiện: chỉ cần **bắt đầu** bằng nguyên âm, hoặc **bắt đầu và kết thúc bằng cùng một ký tự**.
3. Đếm **phụ âm** thay vì nguyên âm (`isalpha` và không thuộc tập nguyên âm), hoặc xét chữ hoa lẫn chữ thường (dùng `.lower()`).
4. Trả về **danh sách** các vowel string (hoặc chỉ số) thay vì số lượng.
5. Đếm các chuỗi mà **mọi** ký tự đều là nguyên âm (dùng `all(c in V for c in w)`).
6. Dữ liệu rất lớn (không đọc hết vào bộ nhớ): duyệt luồng (generator) và đếm.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Duyệt cả mảng + `continue` | so sánh từng nguyên âm | `O(n)` | `O(1)` | chỉ để hiểu đề |
| 2. Cắt đoạn + `for` + `set` | duyệt đúng đoạn | `O(m)` | `O(m)` / `O(1)` | cách nộp, rõ ràng |
| 3. `startswith`/`endswith` + `sum` | một dòng, tuple tiền tố | `O(m)` | `O(m)` | viết ngắn khi thi |
| 4. Prefix count | tiền tố đếm, hiệu hai mốc | `O(n)` + `O(1)`/truy vấn | `O(n)` | nhiều truy vấn đoạn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().vowelStrings(["apple", "banana", "orange", "kiwi"], 0, 3)` bằng bao nhiêu?

**Bài 2 (tính tay):** `words = ["a", "bb", "ee", "aib"]`, `left = 1`, `right = 3`. Lập bảng `i, w, w[0], w[-1], đếm?` và kết luận.

**Bài 3 (điền TODO):** hoàn thành `is_vowel_string(w)` trả về `True` nếu `w` là vowel string.

**Bài 4 (tự viết lại):** viết `count_in_ranges(words, queries)` nhận danh sách truy vấn `[l, r]` và trả về list đáp án, dùng **prefix count** (dựng `pref` một lần).

**Bài 5 (mini challenge):** viết `longest_vowel_run(words)` trả về độ dài **dãy liên tiếp dài nhất** các chuỗi đều là vowel string (đáp án `0` nếu không có).

In [28]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().vowelStrings(["apple", "banana", "orange", "kiwi"], 0, 3))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def is_vowel_string(w):
    vowels = "aeiou"
    # TODO: trả về True nếu ký tự đầu VÀ ký tự cuối của w đều nằm trong vowels
    pass

2


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** một chuỗi chỉ được tính khi **cả** ký tự đầu **và** ký tự cuối đều là nguyên âm. Chuỗi một ký tự có đầu = cuối.

**Mức 2 (vừa):** Bài 3: `return w[0] in vowels and w[-1] in vowels`. Bài 4: `pref[i + 1] = pref[i] + is_vowel_string(words[i])`, mỗi truy vấn `pref[r + 1] - pref[l]`. Bài 5: duyệt một lần, giữ `cur` (độ dài dãy hiện tại) và `best`; gặp vowel string thì `cur += 1`, ngược lại `cur = 0`.

**Mức 3 (gần đáp án):** Bài 1: `apple` (a…e) có, `banana` không (b), `orange` (o…e) có, `kiwi` không ⇒ `2`. Bài 2: `bb` không, `ee` có, `aib` kết thúc `b` nên không ⇒ `1`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
assert Solution().vowelStrings(["apple", "banana", "orange", "kiwi"], 0, 3) == 2

# Bài 2
assert Solution().vowelStrings(["a", "bb", "ee", "aib"], 1, 3) == 1

# Bài 3
def is_vowel_string_answer(w):
    vowels = "aeiou"
    return w[0] in vowels and w[-1] in vowels

assert is_vowel_string_answer("are") is True
assert is_vowel_string_answer("amy") is False
assert is_vowel_string_answer("u") is True

# Bài 4
def count_in_ranges(words, queries):
    pref = [0]
    for w in words:
        pref.append(pref[-1] + (1 if is_vowel_string_answer(w) else 0))
    return [pref[r + 1] - pref[l] for l, r in queries]

ws = ["aba", "bcb", "ece", "aa", "e"]
assert count_in_ranges(ws, [[0, 2], [1, 4], [1, 1]]) == [2, 3, 0]
assert count_in_ranges(ws, [[0, 4]]) == [4]

# Bài 5
def longest_vowel_run(words):
    best = cur = 0
    for w in words:
        if is_vowel_string_answer(w):
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best

assert longest_vowel_run(["aba", "bcb", "ece", "aa", "e"]) == 3
assert longest_vowel_run(["b", "c"]) == 0
assert longest_vowel_run(["a"]) == 1
print("tất cả đáp án bài tập: PASS")

tất cả đáp án bài tập: PASS


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đếm trong đoạn đóng":**

```python
vowels = set("aeiou")
count = 0
for i in range(left, right + 1):           # + 1: đoạn ĐÓNG
    w = words[i]
    if w[0] in vowels and w[-1] in vowels:  # đầu VÀ cuối
        count += 1
return count
```

**Template prefix count (nhiều truy vấn):**

```python
pref = [0]
for w in words:
    pref.append(pref[-1] + (w[0] in vowels and w[-1] in vowels))
return pref[right + 1] - pref[left]
```

**Checklist trước khi submit:**

- [ ] Đoạn là `[left, right]` **đóng** (`right + 1` khi `range`/slicing)?
- [ ] Điều kiện là ký tự đầu **và** ký tự cuối (`and`, không phải `or`)?
- [ ] Viết đủ hai vế `w[0] in V and w[-1] in V` (không phải `w[0] and w[-1] in V`)?
- [ ] Chuỗi một ký tự được xử lý đúng (đầu = cuối)?
- [ ] Tập nguyên âm chỉ gồm `a, e, i, o, u` (không có `y`)?
- [ ] Đã thử đoạn một phần tử, cả mảng, không có vowel string nào?
- [ ] Chữ ký đúng `vowelStrings(self, words: List[str], left: int, right: int) -> int`?